In [1]:
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)


In [6]:
from transformers import AutoTokenizer, AutoModel
MODEL_PATH = r"C:\Users\AI-04\.cache\huggingface\hub\models--Rostlab--prot_bert\snapshots\7a894481acdc12202f0a415dd567f6cfdb698908"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

model = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True)

model.eval()
model.to("cpu")

print("Model and tokenizer loaded offline ✅")


Loading weights: 100%|█████████████████████| 487/487 [00:00<00:00, 880.43it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: C:\Users\AI-04\.cache\huggingface\hub\models--Rostlab--prot_bert\snapshots\7a894481acdc12202f0a415dd567f6cfdb698908
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from

Model and tokenizer loaded offline ✅


In [2]:
import numpy as np
def protbert_embedding(sequences, batch_size=1, max_len=512):
    embs = []
    with torch.inference_mode():  # مهم
        for seq in sequences:
            seq = " ".join(list(seq))
            inputs = tokenizer(
                seq,
                return_tensors="pt",
                truncation=True,
                max_length=max_len
            )
            outputs = model(**inputs)
            hidden = outputs.last_hidden_state  # (1, L, 1024)
            emb = hidden.mean(dim=1).cpu().numpy()
            embs.append(emb)
    return np.vstack(embs)


In [8]:
from Bio import SeqIO
import pandas as pd

def load_fasta_to_df(fasta_path):
    records = []
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq)
        label = 1 if "toxic" in record.description.lower() else 0
        records.append({"sequence": seq, "label": label})
    return pd.DataFrame(records)

train_df = load_fasta_to_df(
    "ToxicPeptidePrediction/ToxicPeptidePrediction-main/protein_train.fa"
)

test_df = load_fasta_to_df(
    "ToxicPeptidePrediction/ToxicPeptidePrediction-main/protein_test.fa"
)

print(train_df.shape, test_df.shape)


(10084, 2) (729, 2)


In [7]:
seq = train_df["sequence"].iloc[0]

print("start")
emb1 = protbert_embedding([seq])
print("done", emb1.shape)


start
done (1, 1024)


In [6]:
import numpy as np
from tqdm import tqdm

def extract_and_save_embeddings(seqs, save_path):
    all_embs = []
    with torch.inference_mode():
        for seq in tqdm(seqs):
            seq = " ".join(list(seq))
            inputs = tokenizer(
                seq,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )
            outputs = model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            all_embs.append(emb)
    all_embs = np.vstack(all_embs)
    np.save(save_path, all_embs)
    return all_embs


In [7]:
#یکبار اجرا
X_train_emb = extract_and_save_embeddings(
    train_df["sequence"].tolist(),
    "protbert_train.npy"
)

print(X_train_emb.shape)



00%|████████████████████████████████████████████████████████████████████████| 10084/10084 [177:36:28<00:00, 63.41s/it]

(10084, 1024)


In [8]:
X_test_emb = extract_and_save_embeddings(
    test_df["sequence"].tolist(),
    "protbert_test.npy"
)

print(X_test_emb.shape)


100%|██████████████████████████████████████████████████████████████████████████████| 729/729 [3:53:07<00:00, 19.19s/it]

(729, 1024)
